# Day 16：非线性核 SVM

目标：在月牙形数据上比较线性核与 RBF 核。核函数隐式计算特征空间的内积，不要求显式创建所有高维特征。

运行前请阅读[环境与运行说明](../docs/setup.md)。本课 `.py` 是教学源文件，配套 Markdown 和 Notebook 自动同步。图形保存到 `outputs/`，设置 `COURSE_SHOW_PLOTS=1` 可显示窗口。


[Python 源文件](Day%2016_Kernel_SVM.py) · [Notebook](Day%2016_Kernel_SVM.ipynb) · [完整课程目录](../docs/curriculum.md)


In [ ]:
from pathlib import Path
import sys

# 脚本从文件位置定位仓库；Notebook 从当前工作目录向上查找。
base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
for candidate in (base, *base.parents):
    if (candidate / "Code" / "course_utils.py").is_file():
        code_dir = str(candidate / "Code")
        if code_dir not in sys.path:
            sys.path.insert(0, code_dir)
        break
else:
    raise FileNotFoundError("找不到课程仓库，请从仓库根目录或 Code 目录启动 Notebook。")
from course_utils import DATA, OUTPUT, finish_plot


## 训练内选择 C 和 gamma

C 控制误分类惩罚，gamma 控制 RBF 相似度的作用范围。二者在训练集交叉验证内联合选择；测试集保留到最后。


In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from course_utils import classification_summary, decision_plot
X, y = make_moons(n_samples=300, noise=0.2, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)
linear = make_pipeline(StandardScaler(), SVC(kernel="linear")).fit(X_train, y_train)
search = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")),
                      {"svc__C": [0.1, 1, 10], "svc__gamma": [0.1, 1, 10]}, cv=5)
search.fit(X_train, y_train)
model = search.best_estimator_
print("Selected parameters:", search.best_params_)
print("Linear test accuracy:", linear.score(X_test, y_test))
y_pred = classification_summary(model, X_test, y_test, "day16")
for name, estimator in [("linear", linear), ("rbf", model)]:
    decision_plot(estimator, X_test, y_test, "day16_" + name, ["Feature 1", "Feature 2"])


## 练习与检查

只在训练/验证集上增加噪声并扩大 gamma，观察过拟合。说明测试图展示模型表现，而不用于继续选择参数。
